### 1. Environment setup

In [ ]:
# Import required packages
import os
import time
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv

# Load S3 credentials
load_dotenv(".env")


In [ ]:
# Setup DuckDB and connect to MicroCeph

# Create a connection to an in-memory DuckDB instance
con = duckdb.connect()

# Install and load the httpfs extension for S3 access
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")

# Create an S3 secret using the credentials from the environment
con.execute(f"""
    CREATE OR REPLACE SECRET microceph_s3 (
        TYPE s3,
        KEY_ID '{os.environ['S3_ACCESS_KEY']}',
        SECRET '{os.environ['S3_SECRET_KEY']}',
        ENDPOINT 'localhost:8080',
        URL_STYLE 'path',
        USE_SSL false
    );
""")

print("DuckDB is connected to MicroCeph S3.")

### 2. Gold layer analysis

#### Taxi demand and trip economics

In [ ]:
t0 = time.perf_counter()

# Group all trips by hour of the day and split them into weekend trips and weekday trips
hourly = con.execute("""
    SELECT pickup_hour,
           SUM(CASE WHEN is_weekend     THEN total_trips ELSE 0 END) AS weekend_trips,
           SUM(CASE WHEN NOT is_weekend THEN total_trips ELSE 0 END) AS weekday_trips
    FROM read_parquet('s3://owning-the-stack/gold/monthly_summary_*.parquet')
    WHERE year = 2025 AND month = 5
    GROUP BY 1 ORDER BY 1
""").df()

t1 = time.perf_counter()

# Average tip percentage by payment type
tips = con.execute("""
    SELECT payment_type_label, ROUND(AVG(avg_tip_percent), 2) AS tip_pct
    FROM read_parquet('s3://owning-the-stack/gold/monthly_summary_*.parquet')
    WHERE year = 2025 AND month = 5
    GROUP BY 1 ORDER BY 2 DESC
""").df()
t2 = time.perf_counter()

# Measure query time
print(f"hourly query: {(t1-t0)*1000:6.1f} ms   ({len(hourly)} rows)")
print(f"tip    query: {(t2-t1)*1000:6.1f} ms   ({len(tips)} rows)")

#### Hourly taxi demand: weekdays vs. weekends

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(hourly['pickup_hour'], hourly['weekday_trips'],
        marker='o', linewidth=2, color='#2563eb', label='Weekday')
ax.plot(hourly['pickup_hour'], hourly['weekend_trips'],
        marker='s', linewidth=2, color='#f59e0b', label='Weekend')

# Shade the classic NYC rush-hour windows
ymax = hourly[['weekday_trips', 'weekend_trips']].values.max()
ax.axvspan(7, 9,  color='#2563eb', alpha=0.08)
ax.axvspan(16, 19, color='#2563eb', alpha=0.08)
ax.text(8,    ymax * 0.95, 'AM rush', ha='center', fontsize=9, color='#1e3a8a')
ax.text(17.5, ymax * 0.95, 'PM rush', ha='center', fontsize=9, color='#1e3a8a')

ax.set_title('NYC Yellow Taxi — Hourly Demand, May 2025',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Hour of day (pickup)')
ax.set_ylabel('Total trips')
ax.set_xticks(range(0, 24))
ax.grid(True, alpha=0.3)
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

#### Average recorded tip percentage by payment type

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

# Sort ascending so the highest-tipping type ends up on top of the barh
ts = tips.sort_values('tip_pct')
colors = ['#9ca3af' if lbl == 'Cash' else '#2563eb'
          for lbl in ts['payment_type_label']]

bars = ax.barh(ts['payment_type_label'], ts['tip_pct'], color=colors)

# Value labels at the end of each bar
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.15, bar.get_y() + bar.get_height() / 2,
            f'{w:.1f}%', va='center', fontsize=9)

ax.set_title('Average Tip % by Payment Type — May 2025',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Average tip (as % of fare)')
ax.set_xlim(0, max(tips['tip_pct']) * 1.15)
ax.grid(True, axis='x', alpha=0.3)

# The data caveat — turns a "why is cash 0%?" question into a talking point
ax.text(0.98, 0.03,
        'Note: TLC does not record cash tips. "Cash" reflects only recorded (card) tips.',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=8, color='#6b7280', style='italic')

plt.tight_layout()
plt.show()

#### Average trip speed by pickup hour

In [ ]:
speed = con.execute("""
    SELECT pickup_hour,
        SUM(CASE WHEN is_weekend     THEN avg_speed_mph*total_trips ELSE 0 END) /
            NULLIF(SUM(CASE WHEN is_weekend     THEN total_trips ELSE 0 END), 0) AS weekend_speed,
        SUM(CASE WHEN NOT is_weekend THEN avg_speed_mph*total_trips ELSE 0 END) /
            NULLIF(SUM(CASE WHEN NOT is_weekend THEN total_trips ELSE 0 END), 0) AS weekday_speed
    FROM read_parquet('s3://owning-the-stack/gold/monthly_summary_*.parquet')
    WHERE year = 2025 AND month = 5
    GROUP BY 1 ORDER BY 1
""").df()

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(speed['pickup_hour'], speed['weekday_speed'],
        marker='o', linewidth=2, color='#2563eb', label='Weekday')
ax.plot(speed['pickup_hour'], speed['weekend_speed'],
        marker='s', linewidth=2, color='#f59e0b', label='Weekend')
ax.axvspan(7,  9,  color='#2563eb', alpha=0.08)
ax.axvspan(16, 19, color='#2563eb', alpha=0.08)
ax.set_title('Average trip speed by hour — May 2025', fontsize=13, fontweight='bold')
ax.set_xlabel('Hour of day'); ax.set_ylabel('Average speed (mph)')
ax.set_xticks(range(0, 24)); ax.grid(True, alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

#### Hourly taxi revenue: weekdays vs. weekends

In [ ]:
rev = con.execute("""
    SELECT pickup_hour,
        SUM(CASE WHEN is_weekend     THEN total_revenue ELSE 0 END) AS weekend_rev,
        SUM(CASE WHEN NOT is_weekend THEN total_revenue ELSE 0 END) AS weekday_rev
    FROM read_parquet('s3://owning-the-stack/gold/monthly_summary_*.parquet')
    WHERE year = 2025 AND month = 5
    GROUP BY 1 ORDER BY 1
""").df()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(rev['pickup_hour'] - 0.2, rev['weekday_rev'], width=0.4,
       color='#2563eb', label='Weekday')
ax.bar(rev['pickup_hour'] + 0.2, rev['weekend_rev'], width=0.4,
       color='#f59e0b', label='Weekend')
ax.set_title('Hourly revenue — May 2025', fontsize=13, fontweight='bold')
ax.set_xlabel('Hour of day'); ax.set_ylabel('Revenue (USD)')
ax.set_xticks(range(0, 24)); ax.grid(True, axis='y', alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

total_weekday = rev['weekday_rev'].sum()
total_weekend = rev['weekend_rev'].sum()
print(f"Weekday revenue: ${total_weekday:,.0f}   Weekend: ${total_weekend:,.0f}")

#### Trip volume by payment type

In [ ]:
payment = con.execute("""
    SELECT payment_type_label, SUM(total_trips) AS trips
    FROM read_parquet('s3://owning-the-stack/gold/monthly_summary_*.parquet')
    WHERE year = 2025 AND month = 5
    GROUP BY 1 ORDER BY 2 DESC
""").df()

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#2563eb', '#9ca3af', '#f59e0b', '#ef4444', '#a855f7', '#10b981']
ax.barh(payment['payment_type_label'], payment['trips'], color=colors)

total = payment['trips'].sum()
for i, (label, trips) in enumerate(zip(payment['payment_type_label'], payment['trips'])):
    pct = 100 * trips / total
    ax.text(trips, i, f'  {pct:.1f}%', va='center', fontsize=9)

ax.set_title('Trip mix by payment type — May 2025', fontsize=13, fontweight='bold')
ax.set_xlabel('Number of trips')
ax.invert_yaxis()
ax.grid(True, axis='x', alpha=0.3)
ax.text(0.98, 0.03,
        'TLC does not record cash tips — that is why avg tip % appears as 0 for cash.',
        transform=ax.transAxes, ha='right', va='bottom',
        fontsize=8, color='#6b7280', style='italic')
plt.tight_layout(); plt.show()

#### Trip distance, duration, and fare by pickup hour

In [ ]:
char = con.execute("""
    SELECT pickup_hour,
        SUM(avg_distance*total_trips)      / SUM(total_trips) AS distance_mi,
        SUM(avg_duration_minutes*total_trips) / SUM(total_trips) AS duration_min,
        SUM(avg_fare*total_trips)          / SUM(total_trips) AS fare_usd
    FROM read_parquet('s3://owning-the-stack/gold/monthly_summary_*.parquet')
    WHERE year = 2025 AND month = 5
    GROUP BY 1 ORDER BY 1
""").df()

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), sharex=True)
for ax, col, title, color in [
    (axes[0], 'distance_mi', 'Avg distance (miles)', '#2563eb'),
    (axes[1], 'duration_min', 'Avg duration (min)',  '#10b981'),
    (axes[2], 'fare_usd',     'Avg fare (USD)',      '#f59e0b'),
]:
    ax.plot(char['pickup_hour'], char[col], marker='o', linewidth=2, color=color)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Hour of day')
    ax.set_xticks(range(0, 24, 3))
    ax.grid(True, alpha=0.3)

fig.suptitle('Trip character by hour — May 2025', fontsize=13, fontweight='bold', y=1.03)
plt.tight_layout(); plt.show()